# 11.12 圖片裡的字也能辨識嗎？


認出「紅色圓形」還不等於會讀圖片中的數字。數字0與1由不同筆畫組成，圖片裡可能還有不同字體、位移或粗細。本節先把兩種最小字形變成像素，讓我們看清楚輸入和正確標籤如何對應。這種從圖中文字讀出字符的任務叫OCR，英文Optical Character Recognition。

前置是[10.1像素tensor](https://birdhackor.github.io/tiny-perceptron-vlm/10.1.html)與[10.5分類監督](https://birdhackor.github.io/tiny-perceptron-vlm/10.5.html)。不依賴電腦字體，我們直接用五行、每行三格畫字：1表示白色筆畫，0表示黑色背景。這裡字串鍵"0"是正確文字標籤，圖裡的數值0則是黑色像素，兩者角色不同。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

glyphs = {"0": ["111", "101", "101", "101", "111"], "1": ["010", "110", "010", "010", "111"]}
for label, rows in glyphs.items():
    image = torch.tensor([[int(c) for c in row] for row in rows], dtype=torch.float32)
    print("文字標籤", label, "像素形狀", tuple(image.shape))
    print(image)

輸入字典存兩種人工字形。`glyphs.items()`每次取出一組「鍵、值」：第一組的鍵是字串"0"，值是它的五行字形清單；`label`接住這個鍵，`rows`接住該清單。下一次則接住標籤"1"與它自己的五行，讓標籤和像素保持成對。內層`for c in row`逐字讀一行的"0"或"1"，`int(c)`轉成數值；外層逐行組成二維清單，最後變成浮點tensor。輸出每張形狀`(5,3)`。標籤0的第一與最後一行全1、中間是[1,0,1]，形成空心框；標籤1上方以中間直線為主，最後一行全1做底座。你可以從打印矩陣直接核對筆畫。

這段只準備資料，沒有訓練辨識器。之後可讓分類頭預測0或1，也可把標籤接成語言模型的目標回答；前者一次選類別，後者逐步輸出文字。真值指資料事先給定的正確標籤，例如字形0的標籤是字串"0"。分類時，選類別0算對、選1算錯；生成回答時，可以要求完整字串與標籤相同，輸出"0"算對，"1"或多出一字的"00"都算錯。兩者用同一份正確標籤，但生成還要核對是否多字、少字。若要沿用本項目的RGB視覺入口，還需把單通道字形放到固定畫布，再複製成三通道；不能直接把(5,3)當成(3,16,16)。

只看同一張字形的成功不表示認得所有字體。訓練與測試可以保留不同位移、筆畫粗細或噪聲條件，看模型是否抓到穩定形狀，而不是背出固定位置。生成規則要保存，讓錯誤可回查原圖與標籤；完整訓練流程可另讀[訓練指南](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)。

字形資料最重要的對應是「這份像素確實寫這個標籤」。如果繪圖器把1畫成0，程式仍會照標籤訓練，梯度不會替作者糾正。小矩陣的優勢是讀者能直接數格子：0的中間三行左右亮、中間暗，1的主豎線在中央。位移可以在外加空白格，而不是把筆畫換成別的字。改變字體後標籤不變，測試的正是對同一字符外觀變化的穩定性；改變字符後則必須改標籤，這兩種資料增強不能混在一起。

[正式OCR實驗](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/ocr.json)另外把0到99的一至兩位數畫進16×16 RGB畫布，白字黑底，用的是每字五格寬、七格高的固定字形，和上面三格寬的小示意不同。每個字串各有水平位移−1、0、1的三張圖，先按字串分成80個訓練家族、10個驗證家族、10個測試家族，再生成240、30、30張圖；同一字串的三種位置不跨組。這是在問「已看過這套字形，能否讀出新的數字組合」，沒有測手寫、另一套字體或文件照片。

這次從直接SFT文字底座與已訓練的視覺編碼器接起，全模型開放更新500次，每次八張，累計11,584個有效回答目標。同一批訓練探針的代價從9.282降到0.809，但驗證只9/30完整讀對，測試更只有2/30，約6.7%。代價下降像練習抄字時越寫越順，換到新編號仍可能抄錯；不能拿熟悉練習紙上的進步代替新題辨識。下一節拆開這批錯字，看看整串計數與字元錯誤各能告訴我們什麼。

練習把標籤1的第一行從"010"改"110"，先畫出新增的左上筆畫，確認文字標籤仍是1，再執行查看第一行矩陣變化。只改一種筆畫，不能同時把標籤改0；我們要觀察字體變化，而不是改變需要辨識的字符。
